# ARC Prize 2026 — ARC-AGI-2 Solver v3

## Architecture Overview

Multi-strategy ensemble: **70+ heuristic solvers**, **object-centric reasoning**, **similarity search**, **LLM self-consistency**, **LoRA test-time training**, **evolutionary synthesis**.

### Key Improvements over v2

| # | Improvement | Description |
|---|------------|-------------|
| 1 | **Grid Embedding & Similarity Search** | Match eval tasks to training tasks; reuse known solutions |
| 2 | **Object-Centric Reasoning** | Detect per-object transformations (translate, scale, color) |
| 3 | **70+ Heuristic Solvers** | Expanded from 38: morphological, sort, compress, grow/shrink, outline, fill |
| 4 | **Test-Time LoRA Training** | Quick LoRA fine-tune Qwen3-4B per task (NVARC-inspired) |
| 5 | **Self-Consistency LLM** | 3 code candidates, pick best by training accuracy |
| 6 | **Improved Evolution** | More mutations, population 15, 4 generations |
| 7 | **Tiered Ensemble** | Heuristics -> Object -> Similarity -> LLM -> LoRA TTT |

### Pipeline
```
1. 70+ heuristic solvers -> best partial-match
2. Object-centric reasoning
3. Similarity search against training tasks
4. LLM self-consistency (3 candidates)
5. Evolutionary synthesis
6. LoRA test-time training (last resort)
7. Ensemble: top-2 distinct predictions
```

### Kaggle Constraints
- GPU: T4/P100 (16GB), no internet, ~12h
- Output: `submission.json` with 2 attempts per task

In [ ]:
import json, os, copy, time, sys, traceback, threading, re, hashlib, random, math
from pathlib import Path
from collections import Counter, defaultdict, deque
from typing import Any, Dict, List, Optional, Tuple, Callable
import numpy as np

_HAS_TORCH = _HAS_TRANSFORMERS = _HAS_BNB = _HAS_PEFT = False
try:
    import torch; _HAS_TORCH = True
except ImportError: pass
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig; _HAS_TRANSFORMERS = True
except ImportError: pass
try:
    import bitsandbytes; _HAS_BNB = True
except ImportError: pass
try:
    from peft import LoraConfig, get_peft_model, TaskType; _HAS_PEFT = True
except ImportError: pass
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kw):
        d = kw.get('desc','')
        if d: print(f"{d}...")
        for x in it: yield x

DATA_DIR = "/kaggle/input/arc-prize-2026-arc-agi-2"
OUTPUT_PATH = "/kaggle/working/submission.json"
MODEL_NAME = "Qwen/Qwen3-4B"
QUANTIZATION_4BIT = True
MAX_NEW_TOKENS = 2048
MAX_INPUT_TOKENS = 30000
CODE_TIMEOUT_SECONDS = 10
MAX_LLM_TIME_SECONDS = 300
NUM_SUBMISSION_ATTEMPTS = 2
EVOLUTIONARY_POPULATION = 15
EVOLUTIONARY_GENERATIONS = 4
LORA_RANK = 8
LORA_ALPHA = 16
LORA_STEPS = 50
LORA_LR = 5e-5

COLOR_NAMES = {0:'black',1:'blue',2:'red',3:'green',4:'yellow',5:'gray',6:'magenta',7:'orange',8:'cyan',9:'brown'}

print(f"[CONFIG] Torch:{_HAS_TORCH} Transformers:{_HAS_TRANSFORMERS} BnB:{_HAS_BNB} PEFT:{_HAS_PEFT}")
if _HAS_TORCH and torch.cuda.is_available():
    print(f"[CONFIG] GPU: {torch.cuda.get_device_name(0)}")

## Grid Utilities

In [ ]:
def grid_to_str(grid, label=''):
    if not grid: return f'{label}: (empty)'
    rows, cols = len(grid), len(grid[0])
    h = f'{label} ({rows}x{cols}):' if label else f'({rows}x{cols}):'
    return h + '\n' + '\n'.join(' '.join(str(v) for v in row) for row in grid)

def grids_equal(a, b):
    if len(a) != len(b): return False
    for ra, rb in zip(a, b):
        if len(ra) != len(rb): return False
        for va, vb in zip(ra, rb):
            if va != vb: return False
    return True

def g2n(grid): return np.array(grid, dtype=np.int64)
def n2g(arr): return arr.astype(int).tolist()

def normalize(result):
    if isinstance(result, list) and all(isinstance(r, list) for r in result):
        try: return [[int(v)%10 for v in row] for row in result]
        except: pass
    if isinstance(result, np.ndarray):
        if result.ndim == 2: return result.astype(int).tolist()
        elif result.ndim == 1: return [result.astype(int).tolist()]
    return None

def grid_hash(grid):
    return hashlib.md5(json.dumps(grid, sort_keys=True).encode()).hexdigest()[:12]

def grid_shape(grid):
    return (len(grid), len(grid[0])) if grid else (0, 0)

def find_components(grid, conn=4):
    arr = g2n(grid); h, w = arr.shape
    vis = np.zeros((h,w), dtype=bool); comps = []
    nb = [(-1,0),(1,0),(0,-1),(0,1)] if conn==4 else [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
    for r in range(h):
        for c in range(w):
            if arr[r,c] != 0 and not vis[r,c]:
                color = int(arr[r,c]); comp = []; stk = [(r,c)]
                while stk:
                    cr, cc = stk.pop()
                    if cr<0 or cr>=h or cc<0 or cc>=w or vis[cr,cc] or arr[cr,cc]!=color: continue
                    vis[cr,cc] = True; comp.append((cr,cc))
                    for dr,dc in nb: stk.append((cr+dr,cc+dc))
                comps.append(comp)
    return comps

def comp_bbox(comp):
    rows = [p[0] for p in comp]; cols = [p[1] for p in comp]
    return (min(rows), min(cols), max(rows), max(cols))

def count_colors(grid):
    return Counter(int(v) for v in g2n(grid).flatten().tolist())

def extract_features(grid):
    arr = g2n(grid); h, w = arr.shape; colors = count_colors(grid)
    nz = {c for c in colors if c != 0}; total = h*w
    nz_cells = sum(colors[c] for c in nz)
    comps = find_components(grid, 4)
    return {'height':h,'width':w,'is_square':h==w,'wider':w>h,'taller':h>w,
            'num_colors':len(nz),'density':nz_cells/max(total,1),
            'num_comp':len(comps),'nz_cells':nz_cells,
            'sym_h':bool(np.array_equal(arr,arr[:,::-1])),'sym_v':bool(np.array_equal(arr,arr[::-1,:])),
            'dominant':colors.most_common(1)[0][0] if colors else 0,'colors':dict(colors)}

def describe_grid(grid, label='Grid'):
    f = extract_features(grid)
    parts = [f'{label}: {f["height"]}x{f["width"]}']
    cdescs = [f'{COLOR_NAMES.get(c,str(c))}({c})x{f["colors"][c]}' for c in sorted(f["colors"]) if c != 0]
    parts.append('Colors: ' + ', '.join(cdescs) if cdescs else 'Colors: all black')
    parts.append(f'Objects: {f["num_comp"]} Shape: {"square" if f["is_square"] else "rect"}')
    return ' | '.join(parts)

print('[UTILS] Grid utilities loaded.')

## Data Loading

In [ ]:
def load_task(fp):
    with open(fp) as f: return json.load(f)

def load_all(data_dir, split='evaluation'):
    d = os.path.join(data_dir, 'data', split); tasks = {}
    if not os.path.isdir(d): return tasks
    for fn in sorted(os.listdir(d)):
        if fn.endswith('.json'): tasks[fn[:-5]] = load_task(os.path.join(d, fn))
    return tasks

def load_arc(data_dir):
    ev = load_all(data_dir, 'evaluation'); tr = load_all(data_dir, 'training')
    print(f'[DATA] {len(ev)} eval, {len(tr)} training tasks.')
    return ev, tr

print('[DATA] Ready.')

## Grid Embedding & Similarity Search (NEW v3)

In [ ]:
def task_embedding(task):
    '''Feature vector from train pairs for similarity matching.'''
    feats = []
    for p in task.get('train', []):
        fi = extract_features(p['input']); fo = extract_features(p['output'])
        feats.extend([fi['height'],fi['width'],fo['height'],fo['width'],
                      fi['num_colors'],fo['num_colors'],fi['num_comp'],fo['num_comp'],
                      fi['density'],fo['density'],int(fi['is_square']),int(fo['is_square']),
                      fi['nz_cells'],fo['nz_cells'],fi['dominant'],fo['dominant']])
    while len(feats) < 180: feats.append(0.0)
    return np.array(feats[:180], dtype=np.float32)

def cos_sim(a, b):
    na, nb_ = np.linalg.norm(a), np.linalg.norm(b)
    return float(np.dot(a,b)/(na*nb_)) if na > 0 and nb_ > 0 else 0.0

class SimilarityIndex:
    def __init__(self): self.embs = {}; self.best_h = {}
    def build(self, train_tasks, heurs):
        print(f'[SIM] Indexing {len(train_tasks)} tasks...')
        for tid, task in train_tasks.items():
            self.embs[tid] = task_embedding(task)
            best = find_best_h(task['train'], 1.0, 1, heurs)
            if best: self.best_h[tid] = (best[0][0], best[0][2])
        print(f'[SIM] Done: {len(self.embs)} indexed, {len(self.best_h)} with solvers.')
    def find_similar(self, task, k=5):
        q = task_embedding(task)
        scores = [(tid, cos_sim(q, emb)) for tid, emb in self.embs.items()]
        scores.sort(key=lambda x: -x[1]); return scores[:k]

sim_idx = SimilarityIndex()
print('[SIM] Ready.')

## Object-Centric Reasoning (NEW v3)

In [ ]:
def analyze_objects(grid):
    '''Decompose grid into objects with features.'''
    arr = g2n(grid); comps = find_components(grid, 4); objs = []
    for comp in comps:
        mr, mc, xr, xc = comp_bbox(comp)
        cells = set(comp); color = int(arr[comp[0][0], comp[0][1]])
        objs.append({'color':color,'size':len(cells),'bbox':(mr,mc,xr,xc),
                     'cr':sum(r for r,c in cells)/max(len(cells),1),
                     'cc':sum(c for r,c in cells)/max(len(cells),1),
                     'cells':cells,'w':xc-mc+1,'h':xr-mr+1})
    return objs

def match_objects(inp, out):
    matches = []; used = set()
    for i, io in enumerate(inp):
        bj, bs = -1, -1
        for j, oo in enumerate(out):
            if j in used: continue
            s = (10 if io['color']==oo['color'] else 0) + (5 if io['size']==oo['size'] else 0) - abs(io['size']-oo['size'])*0.1
            if s > bs: bs, bj = s, j
        if bj >= 0 and bs > 3: matches.append((i, bj)); used.add(bj)
    return matches

def detect_transform(io, oo):
    t = {}
    dr = oo['cr'] - io['cr']; dc = oo['cc'] - io['cc']
    if abs(dr) > 0.1: t['translate_r'] = round(dr, 1)
    if abs(dc) > 0.1: t['translate_c'] = round(dc, 1)
    if io['color'] != oo['color']: t['color_change'] = (io['color'], oo['color'])
    sr = oo['size'] / max(io['size'], 1)
    if abs(sr - 1.0) > 0.1: t['scale'] = round(sr, 2)
    return t

def object_solve(task, tidx=0):
    '''Solve via object-centric transformation detection.'''
    pairs = task['train']; all_t = []; ok = True
    for pair in pairs:
        io = analyze_objects(pair['input']); oo = analyze_objects(pair['output'])
        m = match_objects(io, oo)
        if len(m) != len(io) or len(m) != len(oo): ok = False; break
        pt = [detect_transform(io[i], oo[j]) for i,j in m]; all_t.append(pt)
    if not ok or len(all_t) < 2: return None
    # Check consistency
    ref = all_t[0][0].keys() if all_t[0] else set()
    for pt in all_t[1:]:
        if not pt or set(pt[0].keys()) != ref: return None
        for k in ref:
            if k == 'color_change': continue
            v0, vk = all_t[0][0][k], pt[0][k]
            if isinstance(v0,(int,float)) and isinstance(vk,(int,float)) and abs(v0-vk) > 0.5: return None
    # Apply to test
    ti = task['test'][tidx]['input']; tio = analyze_objects(ti)
    result = g2n(ti).copy()
    for idx in range(min(len(tio), len(all_t[0]))):
        t = all_t[0][idx]; obj = tio[idx]
        if 'color_change' in t:
            oc, nc = t['color_change']
            for r, c in obj['cells']:
                if result[r,c] == oc: result[r,c] = nc
        if 'translate_r' in t or 'translate_c' in t:
            dr = int(round(t.get('translate_r',0))); dc = int(round(t.get('translate_c',0)))
            if abs(dr) <= 30 and abs(dc) <= 30:
                h, w = result.shape; old = list(obj['cells'])
                clr = int(ti[old[0][0], old[0][1]])
                for r, c in old: result[r,c] = 0
                for r, c in old:
                    nr, nc = r+dr, c+dc
                    if 0 <= nr < h and 0 <= nc < w: result[nr,nc] = clr
    nm = normalize(result)
    if nm is None: return None
    # Verify on all train pairs
    for pair in pairs:
        io = analyze_objects(pair['input']); oo = analyze_objects(pair['output'])
        m = match_objects(io, oo)
        if len(m) == len(io) and len(m) == len(oo):
            vr = g2n(pair['input']).copy()
            for idx2 in range(min(len(io), len(m))):
                t = detect_transform(io[m[idx2][0]], oo[m[idx2][1]]); obj = io[m[idx2][0]]
                if 'color_change' in t:
                    oc, nc = t['color_change']
                    for r, c in obj['cells']:
                        if vr[r,c] == oc: vr[r,c] = nc
            if normalize(vr) is None or not grids_equal(normalize(vr), pair['output']): return None
    return nm

print('[OBJ] Object-centric reasoning ready.')

## Expanded Heuristic Library (70+ Solvers)

In [ ]:
# GEOMETRIC (1-8)
def h_identity(g): return copy.deepcopy(g)
def h_rot90(g): return np.rot90(g2n(g),1).tolist()
def h_rot180(g): return np.rot90(g2n(g),2).tolist()
def h_rot270(g): return np.rot90(g2n(g),3).tolist()
def h_fliph(g): return g2n(g)[:,::-1].tolist()
def h_flipv(g): return g2n(g)[::-1,:].tolist()
def h_flipd(g): return g2n(g).T.tolist()
def h_flipad(g): return np.rot90(g2n(g).T).tolist()
# COLOR (9-16)
def h_ccomp(g): return (9-g2n(g)).tolist()
def h_cadd(g,k=1): return ((g2n(g)+k)%10).tolist()
def h_csub(g,k=1): return ((g2n(g)-k)%10).tolist()
def h_fmc(g):
    a=g2n(g);v=a[a>0].tolist()
    if not v: return copy.deepcopy(g)
    mc=Counter(int(x) for x in v).most_common(1)[0][0]; return np.where(a>0,mc,0).tolist()
def h_ecol(g,c=1): return np.where(g2n(g)==c,c,0).tolist()
def h_remap(g):
    a=g2n(g);u=[];m={}
    for v in a.flatten():
        v=int(v)
        if v not in m: m[v]=0 if v==0 else (len(u)+1); u.append(v) if v!=0 else None
    r=np.zeros_like(a)
    for v,nv in m.items(): r[a==v]=nv
    return r.tolist()
def h_ccout(g):
    a=g2n(g);c=sorted(set(int(x) for x in a.flatten() if x!=0))
    return [[int(np.sum(a==x)) for x in c]] if c else [[0]]
def h_icolors(g):
    a=g2n(g);u=sorted(set(int(x) for x in a.flatten() if x!=0))
    if len(u)<=1: return copy.deepcopy(g)
    m={0:0}; [m.__setitem__(u[i],u[(i+1)%len(u)]) for i in range(len(u))]
    r=np.zeros_like(a)
    for o,n in m.items(): r[a==o]=n
    return r.tolist()
# OBJECT (17-22)
def h_crop(g):
    a=g2n(g)
    if a.size==0: return g
    rn=np.where(a.any(1))[0];cn=np.where(a.any(0))[0]
    return a[rn[0]:rn[-1]+1,cn[0]:cn[-1]+1].tolist() if len(rn)>0 and len(cn)>0 else [[0]]
def h_cecrop(g):
    a=g2n(g);cs=sorted(set(int(c) for c in a.flatten() if c!=0))
    if not cs: return [[0]]; subs=[]; mh=0
    for c in cs:
        m=(a==c).astype(int);rn=np.where(m.any(1))[0];cn=np.where(m.any(0))[0]
        if len(rn)==0: continue
        s=m[rn[0]:rn[-1]+1,cn[0]:cn[-1]+1]; subs.append(s); mh=max(mh,s.shape[0])
    if not subs: return [[0]]
    pd=[]
    for s in subs:
        if s.shape[0]<mh: s=np.vstack([s,np.zeros((mh-s.shape[0],s.shape[1]),dtype=int)])
        pd.append(s)
    return np.hstack(pd).tolist()
def h_lcomp(g):
    cs=find_components(g,4)
    if not cs: return [[0]*len(g[0]) for _ in range(len(g))]
    lg=max(cs,key=len);a=np.zeros_like(g2n(g));cl=int(g2n(g)[lg[0][0],lg[0][1]])
    for r,c in lg: a[r,c]=cl
    return a.tolist()
def h_scomp(g):
    cs=find_components(g,4)
    if not cs: return [[0]*len(g[0]) for _ in range(len(g))]
    sm=min(cs,key=len);a=np.zeros_like(g2n(g));cl=int(g2n(g)[sm[0][0],sm[0][1]])
    for r,c in sm: a[r,c]=cl
    return a.tolist()
def h_mtl(g): return h_crop(g)
def h_eac(g): return h_lcomp(g)
# SCALING (23-26)
def h_s2x(g): return np.repeat(np.repeat(g2n(g),2,0),2,1).tolist()
def h_s3x(g): return np.repeat(np.repeat(g2n(g),3,0),3,1).tolist()
def h_shalf(g): a=g2n(g); return a[::2,::2].tolist() if a.shape[0]>=2 and a.shape[1]>=2 else copy.deepcopy(g)
def h_sthird(g): a=g2n(g); return a[::3,::3].tolist() if a.shape[0]>=3 and a.shape[1]>=3 else copy.deepcopy(g)
# TILING (27-30)
def h_t2x2(g): return np.tile(g2n(g),(2,2)).tolist()
def h_t3x3(g): return np.tile(g2n(g),(3,3)).tolist()
def h_reph(g): return np.tile(g2n(g),(1,2)).tolist()
def h_repv(g): return np.tile(g2n(g),(2,1)).tolist()
# PATTERN (31-34)
def h_mhf(g): a=g2n(g); return np.hstack([a,a[:,::-1]]).tolist()
def h_mvf(g): a=g2n(g); return np.vstack([a,a[::-1,:]]).tolist()
def h_m4w(g): a=g2n(g); t=np.hstack([a,a[:,::-1]]); return np.vstack([t,t[::-1,:]]).tolist()
def h_ebord(g): a=g2n(g).copy(); a[1:-1,1:-1]=0 if a.shape[0]>2 and a.shape[1]>2 else None; return a.tolist()
# LOGIC (35-38)
def h_ccden(g): a=g2n(g); return (9-a).tolist() if np.sum(a!=0)/max(a.size,1)>0.5 else copy.deepcopy(g)
def h_csrot(g): a=g2n(g); h,w=a.shape; return np.rot90(a,1).tolist() if h==w else (a[:,::-1].tolist() if w>h else a[::-1,:].tolist())
def h_tbin(g): return (g2n(g)>0).astype(int).tolist()
def h_rzd(g):
    a=g2n(g);c=Counter(int(v) for v in a.flatten())
    return np.where(a==0,c.most_common(1)[0][0],a).tolist() if c else copy.deepcopy(g)

# NEW v3: MORPHOLOGICAL (39-42)
def h_dilate(g):
    a=g2n(g);h,w=a.shape;r=a.copy()
    for row in range(h):
        for col in range(w):
            if a[row,col]!=0:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=row+dr,col+dc
                    if 0<=nr<h and 0<=nc<w and r[nr,nc]==0: r[nr,nc]=a[row,col]
    return r.tolist()
def h_erode(g):
    a=g2n(g);h,w=a.shape;r=np.zeros_like(a)
    for row in range(h):
        for col in range(w):
            if a[row,col]!=0:
                border=False
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=row+dr,col+dc
                    if 0<=nr<h and 0<=nc<w and a[nr,nc]==0: border=True; break
                if not border: r[row,col]=a[row,col]
    return r.tolist()
def h_outline(g):
    a=g2n(g);h,w=a.shape;r=np.zeros_like(a)
    for row in range(h):
        for col in range(w):
            if a[row,col]!=0:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                    nr,nc=row+dr,col+dc
                    if 0<=nr<h and 0<=nc<w and a[nr,nc]==0: r[row,col]=a[row,col]; break
    return r.tolist()
def h_fill_enc(g):
    a=g2n(g);h,w=a.shape;vis=np.zeros((h,w),dtype=bool);q=deque()
    for r in range(h):
        for c in range(w):
            if (r==0 or r==h-1 or c==0 or c==w-1) and a[r,c]==0: q.append((r,c)); vis[r,c]=True
    while q:
        r,c=q.popleft()
        for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr,nc=r+dr,c+dc
            if 0<=nr<h and 0<=nc<w and not vis[nr,nc] and a[nr,nc]==0: vis[nr,nc]=True; q.append((nr,nc))
    res=a.copy()
    for r in range(h):
        for c in range(w):
            if a[r,c]==0 and not vis[r,c]:
                for dr,dc in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                    nr,nc=r+dr,c+dc
                    if 0<=nr<h and 0<=nc<w and a[nr,nc]!=0: res[r,c]=a[nr,nc]; break
    return res.tolist()
# SORT (43-46)
def h_srs(g): a=g2n(g); return a[np.argsort(a.sum(1))].tolist()
def h_scs(g): a=g2n(g); return a[:,np.argsort(a.sum(0))].tolist()
def h_sru(g): a=g2n(g); return a[np.argsort([len(set(r)) for r in a])].tolist()
def h_scu(g): a=g2n(g); return a[:,np.argsort([len(set(a[:,c])) for c in range(a.shape[1])])].tolist()
# COMPRESS (47-50)
def h_cprs(g): a=g2n(g);m=a.any(1); return a[m].tolist() if m.any() else [[0]]
def h_cpcl(g): a=g2n(g);m=a.any(0); return a[:,m].tolist() if m.any() else [[0]]
def h_enzr(g): a=g2n(g); return a[a.any(1)].tolist()
def h_enzc(g): a=g2n(g); return a[:,a.any(0)].tolist()
# DEDUP (51-52)
def h_ddr(g):
    a=g2n(g);seen=[];r=[]
    for row in a: k=tuple(row.tolist()); r.append(row) if k not in seen else None; seen.append(k) if k not in seen else None
    return np.array(r,dtype=int).tolist() if r else [[0]]
def h_ddc(g):
    a=g2n(g);seen=[];keep=[]
    for c in range(a.shape[1]): k=tuple(a[:,c].tolist()); keep.append(c) if k not in seen else None; seen.append(k) if k not in seen else None
    return a[:,keep].tolist() if keep else [[0]]
# SWAP (53-56)
def h_swc(g,c1=1,c2=2): a=g2n(g).copy();m1=a==c1;m2=a==c2;a[m1]=c2;a[m2]=c1;return a.tolist()
def h_repc(g,o=1,n=2): a=g2n(g).copy();a[a==o]=n;return a.tolist()
def h_swrtb(g): a=g2n(g);h=a.shape[0]; return np.vstack([a[h//2:],a[:h//2]]).tolist() if h>=2 else a.tolist()
def h_swclr(g): a=g2n(g);w=a.shape[1]; return np.hstack([a[:,w//2:],a[:,:w//2]]).tolist() if w>=2 else a.tolist()
# STACK (57-58)
def h_stkv(g):
    cs=find_components(g,4)
    if not cs: return [[0]]
    parts=[g for _,c in sorted([(comp_bbox(c)[0],comp_bbox(c)[1],c) for c in cs]) for _,_,c in [()] and [extract_comp(g,c)]]; parts=[]
    for c in sorted(cs,key=lambda x:(comp_bbox(x)[0],comp_bbox(x)[1])):
        mr,mc,xr,xc=comp_bbox(c);s=g2n(g)[mr:xr+1,mc:xc+1];parts.append(s)
    mw=max((p.shape[1] for p in parts),default=0);r=[]
    for p in parts:
        for row in p: r.append(list(row)+[0]*(mw-len(row)))
        r.append([0]*mw)
    if r: r.pop()
    return r if r else [[0]]
def extract_comp(g,c):
    mr,mc,xr,xc=comp_bbox(c);return g2n(g)[mr:xr+1,mc:xc+1].tolist()
def h_stkh(g):
    cs=find_components(g,4)
    if not cs: return [[0]]; parts=[]
    for c in sorted(cs,key=lambda x:(comp_bbox(x)[0],comp_bbox(x)[1])):
        mr,mc,xr,xc=comp_bbox(c);s=g2n(g)[mr:xr+1,mc:xc+1];parts.append(s)
    mh=max((len(p) for p in parts),default=0);r=[]
    for ri in range(mh):
        row=[]
        for p in parts:
            if ri<len(p): row.extend(list(p[ri]))
            else: row.extend([0]*(p.shape[1] if p.ndim==2 else 0))
            row.append(0)
        row.pop(); r.append(row)
    return r if r else [[0]]
# INTERLEAVE (59-60)
def h_ilr(g): a=g2n(g);h=a.shape[0]; t=a[:h//2];b=a[h//2:]; r=[]
    for i in range(max(len(t),len(b))):
        if i<len(t): r.append(t[i])
        if i<len(b): r.append(b[i])
    return np.array(r,dtype=int).tolist()
def h_ilc(g): a=g2n(g);w=a.shape[1]; l=a[:,:w//2];ri=a[:,w//2:]; cols=[]
    for i in range(max(l.shape[1],ri.shape[1])):
        if i<l.shape[1]: cols.append(l[:,i])
        if i<ri.shape[1]: cols.append(ri[:,i])
    return np.column_stack(cols).tolist()
# GROW/SHRINK/CENTER (61-64)
def h_grow(g): return h_dilate(g)
def h_shrnk(g): return h_erode(g)
def h_center(g):
    cr=h_crop(g)
    if cr==[[0]]: return copy.deepcopy(g)
    cg=g2n(cr);oh,ow=grid_shape(g);ch,cw=cg.shape
    r=np.zeros((oh,ow),dtype=int);sr=(oh-ch)//2;sc=(ow-cw)//2
    er=min(sr+ch,oh);ec=min(sc+cw,ow);r[sr:er,sc:ec]=cg[:er-sr,:ec-sc]
    return r.tolist()
def h_trans(g,dr=1,dc=0):
    a=g2n(g);h,w=a.shape;r=np.zeros_like(a)
    for row in range(h):
        for col in range(w):
            nr,nc=row+dr,col+dc
            if 0<=nr<h and 0<=nc<w: r[nr,nc]=a[row,col]
    return r.tolist()
# SPECIAL (65-70)
def h_diag(g): a=g2n(g);s=min(a.shape); return [[int(a[i,i]) for i in range(s)]]
def h_adiag(g): a=g2n(g);h,w=a.shape; s=min(h,w); return [[int(a[i,w-1-i]) for i in range(s)]]
def h_rollh(g):
    a=g2n(g);h,w=a.shape
    for p in range(1,w//2+1):
        if w%p==0 and np.all(a[:,i:i+p]==a[:,:p] for i in range(p,w,p)):
            return a[:,:p].tolist()
    return copy.deepcopy(g)
def h_rollv(g):
    a=g2n(g);h,w=a.shape
    for p in range(1,h//2+1):
        if h%p==0 and np.all(a[i:i+p,:]==a[:p,:] for i in range(p,h,p)):
            return a[:p,:].tolist()
    return copy.deepcopy(g)
def h_dist(g):
    cs=find_components(g,4)
    if not cs or len(cs)<2: return copy.deepcopy(g)
    a=g2n(g);h,w=a.shape;r=np.zeros_like(a);sc=sorted(cs,key=lambda c:comp_bbox(c)[1]);sp=w//len(sc)
    for idx,comp in enumerate(sc):
        clr=int(a[comp[0][0],comp[0][1]]);mr,mc,xr,xc=comp_bbox(comp);oh=xr-mr+1;ow=xc-mc+1
        nc=min(idx*sp,w-ow)
        for row,col in comp: nr=row-mr; ncol=col-mc+nc
        if 0<=nr<h and 0<=ncol<w: r[nr,ncol]=clr
    return r.tolist()
def h_tile_detect(g):
    a=g2n(g);h,w=a.shape
    for th in range(1,h//2+1):
        for tw in range(1,w//2+1):
            if h%th==0 and w%tw==0:
                tile=a[:th,:tw]; ok=True
                for r in range(0,h,th):
                    for c in range(0,w,tw):
                        if not np.array_equal(a[r:r+th,c:c+tw],tile): ok=False; break
                    if not ok: break
                if ok: return tile.tolist()
    return copy.deepcopy(g)

print(f'[HEURISTICS] 70 solvers defined.')

## Heuristic Registration & Scoring

In [ ]:
BASE_H = [
    ('identity',h_identity),('rot90',h_rot90),('rot180',h_rot180),('rot270',h_rot270),
    ('fliph',h_fliph),('flipv',h_flipv),('flipd',h_flipd),('flipad',h_flipad),
    ('ccomp',h_ccomp),('icolors',h_icolors),('fmc',h_fmc),('remap',h_remap),('ccout',h_ccout),
    ('crop',h_crop),('cecrop',h_cecrop),('lcomp',h_lcomp),('scomp',h_scomp),
    ('s2x',h_s2x),('s3x',h_s3x),('shalf',h_shalf),('sthird',h_sthird),
    ('t2x2',h_t2x2),('t3x3',h_t3x3),('reph',h_reph),('repv',h_repv),
    ('mhf',h_mhf),('mvf',h_mvf),('m4w',h_m4w),('ebord',h_ebord),
    ('tbin',h_tbin),('rzd',h_rzd),
    ('dilate',h_dilate),('erode',h_erode),('outline',h_outline),('fill_enc',h_fill_enc),
    ('srs',h_srs),('scs',h_scs),('sru',h_sru),('scu',h_scu),
    ('cprs',h_cprs),('cpcl',h_cpcl),('enzr',h_enzr),('enzc',h_enzc),
    ('ddr',h_ddr),('ddc',h_ddc),
    ('swc12',lambda g:h_swc(g,1,2)),('swc13',lambda g:h_swc(g,1,3)),('swc23',lambda g:h_swc(g,2,3)),
    ('swrtb',h_swrtb),('swclr',h_swclr),
    ('stkv',h_stkv),('stkh',h_stkh),('ilr',h_ilr),('ilc',h_ilc),
    ('grow',h_grow),('shrnk',h_shrnk),('center',h_center),
    ('td1',lambda g:h_trans(g,1,0)),('tu1',lambda g:h_trans(g,-1,0)),
    ('tr1',lambda g:h_trans(g,0,1)),('tl1',lambda g:h_trans(g,0,-1)),
    ('diag',h_diag),('adiag',h_adiag),('rollh',h_rollh),('rollv',h_rollv),
    ('dist',h_dist),('tiledet',h_tile_detect),
    ('ccden',h_ccden),('csrot',h_csrot),
]

for c in range(1,10): BASE_H.append((f'ecol{c}',lambda g,c=c:h_ecol(g,c)))
for k in range(1,10): BASE_H.append((f'cadd{k}',lambda g,k=k:h_cadd(g,k))); BASE_H.append((f'csub{k}',lambda g,k=k:h_csub(g,k)))
for c1 in range(1,10):
    for c2 in range(c1+1,10): BASE_H.append((f'swc{c1}{c2}',lambda g,c1=c1,c2=c2:h_swc(g,c1,c2)))

COMP_H = []
def mkcomp(fns):
    def fn(g):
        r=copy.deepcopy(g)
        for f in fns:
            r=normalize(f(r))
            if r is None: return None
        return r
    return fn

COMP_DEFS = [
    ('crop_rot90',[h_crop,h_rot90]),('crop_rot180',[h_crop,h_rot180]),
    ('crop_fliph',[h_crop,h_fliph]),('crop_flipv',[h_crop,h_flipv]),
    ('crop_s2x',[h_crop,h_s2x]),('crop_s3x',[h_crop,h_s3x]),
    ('lcomp_crop',[h_lcomp,h_crop]),
    ('crop_mhf',[h_crop,h_mhf]),('crop_mvf',[h_crop,h_mvf]),
    ('rot90_crop',[h_rot90,h_crop]),('fliph_crop',[h_fliph,h_crop]),('flipv_crop',[h_flipv,h_crop]),
    ('tbin_crop',[h_tbin,h_crop]),('remap_crop',[h_remap,h_crop]),
    ('crop_dilate',[h_crop,h_dilate]),('crop_erode',[h_crop,h_erode]),
    ('crop_outline',[h_crop,h_outline]),('crop_fillenc',[h_crop,h_fill_enc]),
    ('outline_crop',[h_outline,h_crop]),('dilate_crop',[h_dilate,h_crop]),
    ('erode_crop',[h_erode,h_crop]),('fillenc_crop',[h_fill_enc,h_crop]),
    ('crop_cprs',[h_crop,h_cprs]),('crop_cpcl',[h_crop,h_cpcl]),
    ('crop_srs',[h_crop,h_srs]),('crop_scs',[h_crop,h_scs]),
    ('s2x_crop',[h_s2x,h_crop]),('s3x_crop',[h_s3x,h_crop]),
    ('crop_remap_rot',[h_crop,h_remap,h_rot90]),('crop_ccomp',[h_crop,h_ccomp]),
    ('tbin_dilate_crop',[h_tbin,h_dilate,h_crop]),
    ('tbin_erode_crop',[h_tbin,h_erode,h_crop]),
    ('lcomp_outline_crop',[h_lcomp,h_outline,h_crop]),
    ('dilate_outline_crop',[h_dilate,h_outline,h_crop]),
    ('crop_rot90_crop',[h_crop,h_rot90,h_crop]),
    ('crop_fliph_crop',[h_crop,h_fliph,h_crop]),
    ('crop_flipv_crop',[h_crop,h_flipv,h_crop]),
    ('remap_s2x_crop',[h_remap,h_s2x,h_crop]),
    ('crop_fill_outline',[h_crop,h_fill_enc,h_outline,h_crop]),
]

for name,fns in COMP_DEFS: COMP_H.append((name,mkcomp(fns)))

ALL_H = BASE_H + COMP_H

def score_h(fn, pairs):
    p=0;t=len(pairs)
    for pair in pairs:
        try:
            r=normalize(fn(pair['input']))
            if r is not None and grids_equal(r,pair['output']): p+=1
        except: continue
    return (p/t if t>0 else 0.0,p,t)

def find_best_h(pairs, min_s=1.0, max_r=5, heurs=None):
    hl=heurs if heurs else ALL_H; results=[]
    for name,fn in hl:
        s,p,t=score_h(fn,pairs)
        if s>=min_s: results.append((name,fn,s,p,t))
    results.sort(key=lambda x:(-x[2],-x[3],x[0])); return results[:max_r]

def find_partial_h(pairs, max_r=10):
    results=[]
    for name,fn in ALL_H:
        s,p,t=score_h(fn,pairs)
        if s>0: results.append((name,fn,s,p,t))
    results.sort(key=lambda x:(-x[2],-x[3],x[0])); return results[:max_r]

print(f'[HEURISTICS] Total: {len(ALL_H)} ({len(BASE_H)} base + {len(COMP_H)} composite)')

## LLM Engine + Test-Time Training

In [ ]:
class LLMEngine:
    def __init__(self): self.model=None;self.tokenizer=None;self._loaded=False;self._tried=False;self._lora_on=False;self._base=None
    def avail(self):
        if self._tried: return self.model is not None
        if not _HAS_TORCH or not _HAS_TRANSFORMERS: return False
        try: return torch.cuda.is_available()
        except: return False
    def load(self):
        if self._loaded: return; self._tried=True
        print(f'[LLM] Loading {MODEL_NAME}...'); t0=time.time()
        try:
            kw={'torch_dtype':torch.float16,'device_map':'auto','trust_remote_code':True}
            if QUANTIZATION_4BIT and _HAS_BNB:
                kw['quantization_config']=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
            self.tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME,trust_remote_code=True)
            if self.tokenizer.pad_token is None: self.tokenizer.pad_token=self.tokenizer.eos_token
            self.model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,**kw)
            self.model.eval(); self._loaded=True
            print(f'[LLM] Loaded in {time.time()-t0:.1f}s')
        except Exception as e: print(f'[LLM] Failed: {e}'); self.model=None; self.tokenizer=None
    def generate(self,msgs,temp=0.3,mx=MAX_NEW_TOKENS):
        if not self._loaded: self.load()
        if self.model is None: return None
        try:
            txt=self.tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
            inp=self.tokenizer(txt,return_tensors='pt',truncation=True,max_length=MAX_INPUT_TOKENS).to(self.model.device)
            with torch.no_grad():
                out=self.model.generate(**inp,max_new_tokens=mx,temperature=temp,top_p=0.92,do_sample=temp>0,pad_token_id=self.tokenizer.pad_token_id,eos_token_id=self.tokenizer.eos_token_id)
            return self.tokenizer.decode(out[0][inp['input_ids'].shape[1]:],skip_special_tokens=True)
        except Exception as e: print(f'[LLM] Gen err: {e}'); return None
    def ttt(self,pairs,steps=LORA_STEPS,lr=LORA_LR):
        if not _HAS_PEFT or not self._loaded: return False
        try:
            print(f'[TTT] LoRA {steps} steps lr={lr}')
            from torch.utils.data import Dataset
            from transformers import TrainingArguments, Trainer
            class AD(Dataset):
                def __init__(self,p,tok): self.p=p;self.tok=t
                def __len__(self): return len(self.p)*3
                def __getitem__(self,i):
                    pr=self.p[i%len(self.p)]
                    inp='\n'.join(' '.join(str(v) for v in row) for row in pr['input'])
                    out='\n'.join(' '.join(str(v) for v in row) for row in pr['output'])
                    txt=f'Input:\n{inp}\nOutput:\n{out}'
                    enc=self.tok(txt,truncation=True,max_length=2048,padding='max_length',return_tensors='pt')
                    item={k:v.squeeze(0) for k,v in enc.items()}; item['labels']=item['input_ids'].clone(); return item
            if not self._lora_on:
                self._base=self.model
                lc=LoraConfig(r=LORA_RANK,lora_alpha=LORA_ALPHA,target_modules=['q_proj','v_proj'],lora_dropout=0.05,bias='none',task_type=TaskType.CAUSAL_LM)
                self.model=get_peft_model(self.model,lc); self._lora_on=True
            ds=AD(pairs,self.tokenizer)
            ta=TrainingArguments(output_dir='/kaggle/working/ttt_tmp',num_train_epochs=1,per_device_train_batch_size=1,max_steps=steps,learning_rate=lr,logging_steps=10,save_strategy='no',bf16=False,fp16=True,report_to='none',gradient_accumulation_steps=4)
            Trainer(model=self.model,args=ta,train_dataset=ds).train()
            print('[TTT] Done'); return True
        except Exception as e: print(f'[TTT] Err: {e}'); return False

llm=LLMEngine()
print(f'[LLM] Ready: {llm.avail()}')

## Self-Consistency LLM Solver

In [ ]:
P1_SYS='You analyze ARC grid puzzles. Describe transformation rules precisely. Think step by step. Colors: 0=black,1=blue,2=red,3=green,4=yellow,5=gray,6=magenta,7=orange,8=cyan,9=brown.'
P2_SYS='Write a single transform(input_grid) function. input_grid is list of lists of ints 0-9. Return same format. Use numpy as np. Only output code, no explanation.'

def build_prompt(task,tidx=0):
    parts=[]
    for i,p in enumerate(task['train']):
        parts.append(describe_grid(p['input'],f'Ex{i+1} In')); parts.append(grid_to_str(p['input']))
        parts.append(describe_grid(p['output'],f'Ex{i+1} Out')); parts.append(grid_to_str(p['output']))
    parts.append(describe_grid(task['test'][tidx]['input'],'Test In'))
    parts.append(grid_to_str(task['test'][tidx]['input']))
    return '\n'.join(parts)

def extract_code(out):
    if not out: return None
    fm=re.search(r'```(?:python)?\s*
?(.*?)```',out,re.DOTALL)
    if fm:
        b=fm.group(1).strip()
        if 'def transform' in b:
            fn=re.search(r'(def\s+transform\s*\(\s*input_grid\s*\)\s*:.*?)(?=
def\s|\Z)',b,re.DOTALL)
            if fn: return fn.group(1).strip()
            return b
    fm=re.search(r'(def\s+transform\s*\(\s*input_grid\s*\)\s*:.*?)(?=
def\s|\Z)',out,re.DOTALL)
    if fm: return fm.group(1).strip()
    idx=out.find('def transform')
    return out[idx:].strip() if idx>=0 else None

def llm_sc(task,engine,tidx=0,nc=3,mx_time=MAX_LLM_TIME_SECONDS):
    if not engine.avail(): return []
    pairs=task['train']; prompt=build_prompt(task,tidx); results=[]; t0=time.time()
    analysis=engine.generate([{'role':'system','content':P1_SYS},{'role':'user','content':f'Analyze:\n{prompt}'}],0.3,1024) or ''
    for ci in range(nc):
        if time.time()-t0>mx_time: break
        raw=engine.generate([{'role':'system','content':P2_SYS},{'role':'user','content':f'Analysis:\n{analysis}\nExamples:\n{prompt}\nWrite transform.'}],0.2+ci*0.2,MAX_NEW_TOKENS)
        if not raw: continue
        code=extract_code(raw)
        if not code: continue
        s,p,t=verify_code(code,pairs)
        if s>0: results.append((code,s,p))
    seen={}
    for c,s,p in results:
        h=hashlib.md5(c.encode()).hexdigest()[:12]
        if h not in seen or s>seen[h][1]: seen[h]=(c,s,p)
    return sorted(seen.values(),key=lambda x:-x[1])

def safe_exec(code,grid,timeout=CODE_TIMEOUT_SECONDS):
    rc=[];ec=[]
    def worker(c,g,r,e):
        try:
            ns={'np':np,'copy':copy,'math':math,'Counter':Counter,'defaultdict':defaultdict,'deque':deque,'set':set,'sorted':sorted,'len':len,'range':range,'enumerate':enumerate,'zip':zip,'list':list,'int':int,'float':float,'bool':bool,'max':max,'min':min,'abs':abs,'sum':sum,'any':any,'all':all,'print':lambda *a,**k:None,'__builtins__':__builtins__}
            exec(c,ns); fn=ns.get('transform')
            if fn is None: e.append('no fn'); return
            r.append(fn(copy.deepcopy(g)))
        except Exception as ex: e.append(str(ex))
    t=threading.Thread(target=worker,args=(code,grid,rc,ec),daemon=True); t.start(); t.join(timeout)
    if t.is_alive() or ec or not rc: return None
    return normalize(rc[0])

def verify_code(code,pairs):
    p=0;t=len(pairs)
    for pair in pairs:
        r=safe_exec(code,pair['input'])
        if r is not None and grids_equal(r,pair['output']): p+=1
    return (p/t if t>0 else 0.0,p,t)

print('[LLM-SOLVE] Ready.')

## Evolutionary Synthesis (v3)

In [ ]:
class Mutator:
    def mut_num(self,code):
        ms=[]
        for orig in [str(i) for i in range(10)]:
            for repl in [str(j) for j in range(10) if j!=int(orig)][:2]:
                p=re.compile(r'(?<!\w)'+re.escape(orig)+r'(?!\w)'); m=p.sub(repl,code)
                if m!=code: ms.append(m)
        return ms
    def mut_op(self,code):
        ms=[]
        for o,n in [('[:,::-1]','[::-1,:]'),('[::-1,:]','[:,::-1]'),('rot90(g,1)','rot90(g,3)'),('rot90(g,3)','rot90(g,1)'),('rot90(g,2)','g'),('.T',''),('> 0','== 0'),('== 0','> 0'),('max','min'),('min','max')]:
            if o in code: ms.append(code.replace(o,n,1))
        return ms
    def mut_col(self,code):
        ms=[]
        for c in range(10):
            for c2 in range(c+1,10):
                for pat in [f'== {c}',f'!= {c}',f'({c})']:
                    if pat in code: m=code.replace(pat,pat.replace(str(c),str(c2)),1)
                    if m!=code: ms.append(m)
                if len(ms)>=3: break
            if len(ms)>=3: break
        return ms[:3]
    def mut_logic(self,code): return [m for m in [code.replace('>','<',1),code.replace('<','>',1)] if m!=code][:2]
    def mut_swaplines(self,code):
        lines=code.split('\n'); fnl=[(i,l) for i,l in enumerate(lines) if l.strip() and not l.strip().startswith('def ')]
        if len(fnl)<2: return []
        ms=[]
        for _ in range(2):
            i1,l1=random.choice(fnl); i2,l2=random.choice(fnl)
            if i1!=i2: nl=list(lines); nl[i1],nl[i2]=l2,l1; ms.append('\n'.join(nl))
        return ms
    def crossover(self,c1,c2):
        cs=[]
        def eh(c): return [m.group(1) for m in re.finditer(r'(def\s+(\w+)\s*\([^)]*\)\s*:.*?)(?=
def\s+transform|$)',c,re.DOTALL) if m.group(2)!='transform']
        def et(c): m=re.search(r'(def\s+transform\s*\([^)]*\)\s*:.*?$)',c,re.DOTALL); return m.group(1) if m else None
        h1,h2=eh(c1),eh(c2); t1,t2=et(c1),et(c2)
        if t1 and h2: cs.append('\n'.join(h2)+'\n'+t1)
        if t2 and h1: cs.append('\n'.join(h1)+'\n'+t2)
        return cs

def evolve(init,pairs,pop=EVOLUTIONARY_POPULATION,gen=EVOLUTIONARY_GENERATIONS):
    if not init: return []
    mu=Mutator(); pop_={}
    for c,s,p in init: pop_[hashlib.md5(c.encode()).hexdigest()[:12]]=(c,s,p)
    for g in range(gen):
        nc=[]
        for c,s,p in list(pop_.values()):
            if s>=1.0: continue
            for m in mu.mut_num(c)[:2]: nc.append(m)
            for m in mu.mut_op(c)[:2]: nc.append(m)
            for m in mu.mut_col(c)[:1]: nc.append(m)
            for m in mu.mut_logic(c)[:1]: nc.append(m)
            for m in mu.mut_swaplines(c)[:1]: nc.append(m)
        top=[c for c,_,_ in sorted(pop_.values(),key=lambda x:-x[1])][:5]
        for i in range(len(top)):
            for j in range(i+1,len(top)): nc.extend(mu.crossover(top[i],top[j]))
        for cand in nc:
            h=hashlib.md5(cand.encode()).hexdigest()[:12]
            if h not in pop_:
                s,p,t=verify_code(cand,pairs)
                if s>0: pop_[h]=(cand,s,p)
        if len(pop_)>pop: pop_=dict(sorted(pop_.items(),key=lambda x:-x[1][1])[:pop])
        if any(s>=1.0 for s,_ in [(v[1],v[2]) for v in pop_.values()]): break
    return sorted(pop_.values(),key=lambda x:(-x[1],-x[2]))

print(f'[EVOLVE] Ready (pop={EVOLUTIONARY_POPULATION}, gen={EVOLUTIONARY_GENERATIONS})')

## Tiered Ensemble Task Solver

In [ ]:
def safe_apply(fn,grid):
    try: return normalize(fn(copy.deepcopy(grid)))
    except: return None

def solve_task(task,engine,tidx=0,tid='',si=None):
    stats={'tid':tid,'strat':'none','hscore':0.0,'llm':0,'evo':0,'total':0}
    pairs=task['train']; ti=task['test'][tidx]['input']; cands=[]
    # TIER 1: Heuristics
    bh=find_best_h(pairs,1.0,5)
    if bh: stats['hscore']=bh[0][2]; stats['strat']='heuristic'
    for nm,fn,sc,ps,tt in bh:
        r=safe_apply(fn,ti)
        if r: cands.append((r,sc,f'h:{nm}'))
    # TIER 2: Object reasoning
    if stats['hscore']<1.0:
        or_=object_solve(task,tidx)
        if or_: cands.append((or_,1.0,'obj')); stats['strat']='object'
    # TIER 3: Similarity
    if stats['hscore']<1.0 and si and si.best_h:
        for stid,sim in si.find_similar(task,3):
            if stid in si.best_h and sim>0.7:
                for nm,fn,_,_,_ in find_best_h(pairs,1.0,1):
                    if nm==si.best_h[stid][0]:
                        r=safe_apply(fn,ti)
                        if r: cands.append((r,score_h(fn,pairs)[0],f'sim:{stid}'))
                        break
    # TIER 4: LLM
    lcodes=[]
    if engine.avail() and stats['hscore']<1.0:
        bgt=MAX_LLM_TIME_SECONDS if not any(s=='object' for _,s,_ in cands) else 120
        lcodes=llm_sc(task,engine,tidx,3,bgt); stats['llm']=len(lcodes)
        for c,s,p in lcodes:
            r=safe_exec(c,ti)
            if r: cands.append((r,s,'llm'))
    # TIER 5: Evolution
    if lcodes:
        ev=evolve(lcodes,pairs); stats['evo']=len(ev)
        for c,s,p in ev:
            r=safe_exec(c,ti)
            if r: cands.append((r,s,'evo'))
    # TIER 6: TTT
    if stats['hscore']<1.0 and engine.avail() and not any(s>=1.0 for _,s,_ in cands):
        try:
            engine.ttt(pairs)
            for c,s,p in llm_sc(task,engine,tidx,2,120):
                r=safe_exec(c,ti)
                if r: cands.append((r,s,'ttt'))
        except: pass
    # Ensemble
    cands.sort(key=lambda x:-x[1]); sel=[]; seen=set()
    for g,s,src in cands:
        h=grid_hash(g)
        if h not in seen: seen.add(h); sel.append(g)
        if len(sel)>=NUM_SUBMISSION_ATTEMPTS: break
    while len(sel)<NUM_SUBMISSION_ATTEMPTS: sel.append([[0]*len(ti[0]) for _ in range(len(ti))]); break
    stats['total']=len(cands)
    if any(s>=1.0 for _,s,_ in cands): stats['strat']='solved'
    elif cands: stats['strat']='partial'
    return sel[:NUM_SUBMISSION_ATTEMPTS],stats

def solve_all(eval_tasks,engine,si=None,max_t=None,budget=9*3600-600):
    sub={}; tids=list(eval_tasks.keys())
    if max_t: tids=tids[:max_t]
    total=len(tids); ppt=budget/max(total,1)
    print(f'\nSOLVING {total} tasks ({ppt:.0f}s each)\n')
    gs={'h':0,'obj':0,'llm':0,'par':0,'fb':0,'total':total,'solved':0}
    t0=time.time()
    for idx,tid in enumerate(tqdm(tids,desc='Solving')):
        task=eval_tasks[tid]; nt=len(task['test'])
        if nt>1:
            aa=[]
            for ti in range(nt): aa.extend(solve_task(task,engine,ti,tid,si)[0])
            sub[tid]=aa[:NUM_SUBMISSION_ATTEMPTS*nt]
        else: sub[tid]=solve_task(task,engine,0,tid,si)[0]
        st=solve_task(task,engine,0,tid,si)[1]['strat']
        if st in ('heuristic','object','solved'): gs['solved']+=1; gs['h'] if st=='heuristic' else gs['obj'] if st=='object' else gs['llm']
        gs['llm'] if st=='solved' else None
        if st=='partial': gs['par']+=1
        if (idx+1)%20==0 or idx==total-1:
            el=time.time()-t0; rate=(idx+1)/max(el,1); rem=(total-idx-1)/max(rate,0.001)
            print(f'  [{idx+1}/{total}] Solved:{gs["solved"]} Rate:{rate:.2f}/s ETA:{rem/60:.0f}m')
    with open(OUTPUT_PATH,'w') as f: json.dump(sub,f)
    print(f'\nDone. Solved: {gs["solved"]}/{gs["total"]}. Saved to {OUTPUT_PATH}')
    return sub

print('[SOLVER] Tiered ensemble ready.')

## Execution

In [ ]:
# Load data
eval_tasks, train_tasks = load_arc(DATA_DIR)

# Build similarity index
sim_idx.build(train_tasks, ALL_H)

# Solve all
submission = solve_all(eval_tasks, llm, sim_idx)